In [11]:
import pandas as pd
import numpy as np
from nbconvert import export


In [12]:
#Loading in the roads data file
df1 = pd.read_csv('../data/_roads3.csv')
# print(df1.head())

#Filtering on only the N1 road
dfN1 = df1[df1['road'] == 'N1' and 'N2']


FileNotFoundError: [Errno 2] No such file or directory: '../data/_roads3.csv'

In [ ]:
#Creating the source and sink for the new data frame, based on the minimal and maximal chainage
source_val = dfN1['chainage'].min()
sink_val = dfN1['chainage'].max()
source_row = dfN1[dfN1['chainage'] == source_val].head(1)
sink_row = dfN1[dfN1['chainage'] == sink_val].head(1)
#Creating a new DataFrame with the source and sink
df_proc = pd.concat([source_row, sink_row]).drop_duplicates(subset=['chainage', 'lrp']).sort_values(
    'chainage').reset_index(drop=True)
df_proc




In [ ]:
#Loading in the Bridges data
df_BMMS = pd.read_excel('../data/BMMS_overview.xlsx')
#Selecting the columns that we need
cols = [
    "road",
    "name",
    "LRPName",
    "length",
    "chainage",
    "lat",
    "lon",
    "condition",
]

bridges_BMMS = df_BMMS[cols].copy()
bridges_BMMS = bridges_BMMS[bridges_BMMS['road'] == 'N1']
bridges_BMMS.head(20)
#Preprocessing names of bridges so they are more uniform for later duplicates deletion
bridges_BMMS['name_clean'] = bridges_BMMS['name'].astype(str).str.replace(' ', '').str.upper()
#Rounding the location of bridges in order to delete duplicates that are really close together later
bridges_BMMS['lat_round'] = bridges_BMMS['lat'].round(4)
bridges_BMMS['lon_round'] = bridges_BMMS['lon'].round(4)
bridges_BMMS.info()



In [ ]:
#Dropping duplicates
df_unique = bridges_BMMS.drop_duplicates(subset=['chainage', 'lat_round', 'lon_round'], keep='first')
df_unique.info()
df_unique.reset_index(drop=True, inplace=True)
df_unique
#Filtering on bridges only
df_unique['model_type'] = 'bridge'

export_df = df_unique[['road', 'LRPName', 'model_type', 'name', 'lat', 'lon', 'length', 'condition', 'chainage']]
export_df.rename(columns={'LRPName': 'id'}, inplace=True)
export_df.head(10)

In [ ]:
#Manually adding source and sink
df_proc.loc[df_proc.index[0], 'model_type'] = 'source'
df_proc.loc[df_proc.index[-1], 'model_type'] = 'sink'
df_proc['length'] = 0
df_proc['condition'] = None
df_proc = df_proc[['road', 'lrp', 'model_type', 'name', 'lat', 'lon', 'length', 'condition', 'chainage']]
df_proc.rename(columns={'lrp': 'id'}, inplace=True)

df_proc


In [ ]:
#Adding all bridges to final dataframe
final_df = pd.DataFrame(columns=['road', 'id', 'model_type', 'name', 'lat', 'lon', 'length', 'condition', 'chainage'])
source = df_proc.iloc[[0]]
sink = df_proc.iloc[[-1]]
final_df = pd.concat([source, export_df, sink], ignore_index=True)
final_df = final_df.sort_values('chainage').reset_index(drop=True)
final_df
#Filtering the names in final_df with the help of a regex function that processes the string until the first '('. If after replacing dots, lowercase and strip, the names are exactly the same for two bridges, the bridge is treated as duplicate and is deleted.
final_df = (
    final_df
    .assign(
        prefix=(
            final_df["name"]
            .str.replace('.', '')
            .str.lower()
            .str.strip()
            .str.extract(r"^([^(]+)", expand=False)
        )
    )
    .drop_duplicates(subset="prefix", keep="first")
    .drop(columns="prefix")
)

final_df.head(20)

In [ ]:
    #Adding the links in between the bridges. Their exact locations do not matter and are calculated based on the location of the pre- and succeeding bridge. The length of the link does matter, and is calculated based on the chainage of the two bridges it lies in between.
final_df_with_links = []

for j in range(len(final_df)):
    row = final_df.iloc[j]

    if row['model_type'] == 'source':
        final_df_with_links.append(row)
        continue

    if row['model_type'] == 'sink':
        final_df_with_links.append(row)
        break

    prev = final_df.iloc[j - 1]

    lat_link = (row['lat'] + prev['lat']) / 2
    lon_link = (row['lon'] + prev['lon']) / 2

    link_length = (row['chainage'] - prev['chainage']) * 1000  #Because chainage is in km

    link = {
        'road': row['road'],
        'id': None,
        'model_type': 'link',
        'name': 'Link',
        'lat': lat_link,
        'lon': lon_link,
        'length': link_length,
        'condition': None,
        'chainage': (row['chainage'] + prev['chainage']) / 2
    }

    final_df_with_links.append(pd.Series(link))
    final_df_with_links.append(row)

final_df_with_links = (
    pd.DataFrame(final_df_with_links)
    .reset_index(drop=True)
    .drop(columns='chainage')
)

final_df_with_links['id'] = 100000 + (final_df_with_links.index + 1)

final_df_with_links.head(20)

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(final_df_with_links['lon'], final_df_with_links['lat'])
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(final_df_with_links['lon'], final_df_with_links['lat'], marker='o')
plt.show()
final_df_with_links.to_csv('../data/data_N1.csv', na_rep="None")